Check that I have all the sequencing information I expect\
Krista Longnecker 7 April 2026

In [1]:
import pandas as pd
import os
import pdb
from ftplib import FTP
from tqdm import tqdm

In [2]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from sqlalchemy import update,select
from datetime import datetime

# create a SQLite database engine
SQLALCHEMY_DATABASE_URL = "sqlite:///../test_data/sargasso.db"
#SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_testing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
#this will end up creating a new database everytime, but I need this for testing right now

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

# create a session factory
Session = sessionmaker(bind=engine)

# create a declarative base
Base = declarative_base()

In [3]:
from sqlalchemy import inspect
inspector = inspect(engine)
print(inspector.get_table_names())

['cyverse', 'discrete', 'metabolites', 'metabolitesUntargeted', 'sequencingV1V2', 'sequencingV4_16S', 'sequencingV4_18S']


In [4]:
from sqlalchemy import create_engine, inspect, MetaData, Table
metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)

#reflect the tables so I can work on them
user_seqV4_16S = Table('sequencingV4_16S', metadata_obj, autoload_with=engine)
user_seqV4_18S = Table('sequencingV4_18S', metadata_obj, autoload_with=engine)
user_seqV1V2 = Table('sequencingV1V2', metadata_obj, autoload_with=engine)
user_cy = Table('cyverse',metadata_obj,autoload_with=engine)
user_discrete = Table('discrete',metadata_obj,autoload_with=engine)
user_mtab = Table('metabolites',metadata_obj,autoload_with=engine)
user_mtabUntargeted = Table('metabolitesUntargeted',metadata_obj,autoload_with=engine)

In [5]:
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

#session.query(user_discrete).all()

In [6]:
from sqlalchemy import inspect
mapper = inspect(user_discrete)
column_names = [column.key for column in mapper.columns]
print(column_names)

['id', 'bottleID', 'cruise', 'cast', 'niskin', 'yyyymmdd', 'nominalDepth', 'V1V2data', 'V4_16Sdata', 'V4_18Sdata', 'mtabData', 'mtabDataUntargeted']


In [7]:
session.query(user_discrete).all()

[(1, '1033900707', 'AE1718', '7', '7', '20170913', '40', None, None, None, None, None),
 (2, '1033900708', 'AE1718', '7', '8', '20170913', '40', None, None, None, None, None),
 (3, '1033900709', 'AE1718', '7', '9', '20170913', '60', None, None, None, None, None),
 (4, '1033900710', 'AE1718', '7', '10', '20170913', '60', None, None, None, None, None),
 (5, '1033900711', 'AE1718', '7', '11', '20170913', '80', None, None, None, None, None),
 (6, '1033900712', 'AE1718', '7', '12', '20170913', '80', None, None, None, None, None),
 (7, '1033900713', 'AE1718', '7', '13', '20170913', '100', None, None, None, None, None),
 (8, '1033900714', 'AE1718', '7', '14', '20170913', '100', None, None, None, None, None),
 (9, '1033900715', 'AE1718', '7', '15', '20170913', '120', None, None, None, None, None),
 (10, '1033900716', 'AE1718', '7', '16', '20170913', '120', None, None, None, None, None),
 (11, '1033900717', 'AE1718', '7', '17', '20170913', '140', None, None, None, None, None),
 (12, '1033900718

In [8]:
from sqlalchemy import inspect
mapper = inspect(user_cy)
column_names = [column.key for column in mapper.columns]
print(column_names)

['id', 'filename', 'source', 'V4_16S_found', 'V4_18S_found', 'V1V2_found']


In [9]:
session.query(user_cy).all()

[(1, '1032901101-AE1624-10_S1_L001_R1_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (2, '1032901101-AE1624-10_S1_L001_R2_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (3, '1032901103-AE1624-40_S2_L001_R1_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (4, '1032901103-AE1624-40_S2_L001_R2_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (5, '1032901105-AE1624-80_S3_L001_R1_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (6, '1032901107-AE1624-120_S4_L001_R1_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (7, '1032901107-AE1624-120_S4_L001_R2_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (8, '1032901109-AE1624-160_S5_L001_R1_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle', 'cyANDgoogle'),
 (9, '1032901109-AE1624-160_S5_L001_R2_001.fastq.gz', 'cyANDgoogle', 'cyANDgoogle', '

In [10]:
#updating...
from sqlalchemy import update,select

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_discrete.c.bottleID)
    .where(user_discrete.c.V4_16Sdata == user_cy.c.filename)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
#stmt = update(user_cy).values(source=scalar_subq)
stmt = update(user_cy).values(V4_16S_found=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()  

In [11]:
#is it possible Luis did not find any of the V4_18S files?
from sqlalchemy import update,select

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_discrete.c.bottleID)
    .where(user_discrete.c.V4_18Sdata == user_cy.c.filename)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
#stmt = update(user_cy).values(source=scalar_subq)
stmt = update(user_cy).values(V4_18S_found=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()  

In [12]:
#updating...
from sqlalchemy import update,select

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_discrete.c.bottleID)
    .where(user_discrete.c.V1V2data == user_cy.c.filename)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
#stmt = update(user_cy).values(source=scalar_subq)
stmt = update(user_cy).values(V1V2_found=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()  

In [13]:
# see if this worked
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy import Table, Column, Integer, String, MetaData

metadata = MetaData()
users = Table('cyverse', metadata,
    Column('id', Integer, primary_key=True),
    Column('filename', String),
    Column('source', String),
    Column('V4_16S_found', String),
    Column('V4_18S_found', String),
    Column('V1V2_found', String)              
)

#how to execute a query
stmt = select(users)
with engine.connect() as conn:
    rows = session.execute(stmt).all()
    table_data = [row._mapping for row in rows]
    df = pd.DataFrame(table_data)
    df = df.reindex(columns = ['filename','source','V4_16S_found','V4_18S_found','V1V2_found'])

df.head()
df.to_csv('temp2.csv')